# Machine Learning Cloud Deployment Tutorial Series

Tutor: Daniel Ramandi

# Session 2 : AWS setup from scratch

## What we are doing today

Today is the **infrastructure setup** session.

By the end of this notebook, you will have:

- one project folder uploaded into AWS CloudShell,
- one S3 bucket,
- one SageMaker execution role,
- one Elastic Beanstalk service role,
- one Elastic Beanstalk EC2 instance profile,
- one temporary Elastic Beanstalk environment running the sample app.

At the very end, we will **terminate the temporary Elastic Beanstalk environment** so that we do not leave unnecessary AWS resources running. Elastic Beanstalk itself has no extra service charge, but you do pay for the underlying resources it launches, such as EC2. AWS also notes that after terminating an environment, you can launch a new one again later.

## Before we begin: a few plain-English ideas

If some of these words are new, use these simple meanings for today:

- **EC2** = a cloud computer
- **S3** = a cloud folder for files
- **IAM role** = a permission badge that lets one AWS service talk to another AWS service
- **Elastic Beanstalk (EB)** = a managed way to run a web app on AWS
- **SageMaker** = a managed way to host a machine learning model
- **CloudShell** = a terminal inside the AWS browser console

AWS CloudShell is a browser-based shell that is already authenticated with the same AWS account you used to log in, and AWS CLI is already installed there. CloudShell also has Python, `pip`, `tar`, and `zip`, and it keeps up to **1 GB** of persistent storage in your home directory in each Region.

## Important rule (not just for this tutorial)

When working with AWS, make sure you pick **one AWS Region** and keep everything there. Given we are all using a limited-access AWS account, make sure you always have `ca-central-1` selected for region.

That means:
- your S3 bucket,
- your SageMaker resources,
- your Elastic Beanstalk environment,
- and your CloudShell work

should all stay in the **same Region** (`ca-central-1`).

This matters because SageMaker requires the model artifacts in S3 to be in the same Region as the model or endpoint you create later.

## Step 1 : Open the AWS Console and choose a Region

1. Log in to [AWS](https://ubclthubcourses.awsapps.com/start/#).
2. At the top right of the console, set your AWS Region to `ca-central-1`.
3. Keep using this same Region for the full tutorial.


## Step 2 : Launch CloudShell

CloudShell is the place where we will run AWS and Python commands without installing anything on our own computer.

1. In the AWS Console top bar, click the **CloudShell** icon.
2. Wait for the terminal to open.
3. Make sure it opens in the `ca-central-1` Region you chose in Step 1.


<img src="img/img2_1.png" width="400" height="300">

AWS says CloudShell is a browser-based, pre-authenticated shell, and the AWS credentials from your console session are automatically available there.

```bash
aws sts get-caller-identity
aws configure list
python --version
pwd
```

#### What this check does

- `aws sts get-caller-identity` shows which AWS account/identity you are using
- `aws configure list` helps confirm the Region and configuration
- `python --version` confirms Python is available
- `pwd` shows your current folder

These are just sanity checks so you know where you are before creating resources.

<img src="img/img2_2.png" width="400" height="300">

## Step 3 : Upload your project folder to CloudShell

We want a copy of the project scaffold in CloudShell so it is ready for Session 3.

Because CloudShell uploads **one file at a time** through the UI, AWS recommends uploading multiple files as a single **ZIP file**, then unzipping it inside CloudShell.

### On your own computer

Make sure you have a project folder like this:

```text
spam-deploy/
├── model_build/
│   ├── train_model.ipynb
│   ├── inference.py
│   └── model/
│       ├── model.joblib
│       └── metadata.json
├── deploy_sagemaker.py
└── eb_app/
    ├── app.py
    ├── Procfile
    ├── requirements.txt
    └── templates/
        └── index.html

Zip the whole spam-deploy/ folder into something like:

`spam-deploy.zip`

<img src="img/img3_1.png" width="400" height="300">

**In CloudShell:**
1. Click Actions
2. Choose Upload file
3. Upload spam-deploy.zip

<img src="img/img3_2.png" width="400" height="300">

4. Run the following commands to unzip the folder into a new folder called `tutorial-work`

```bash
mkdir -p ~/tutorial-work
cd ~/tutorial-work
unzip ~/spam-deploy.zip
find spam-deploy -maxdepth 3 -type f | sort

#### What this does

- creates a clean working folder,
- unzips your uploaded project,
- and lists the files so you can confirm they arrived.

AWS documents this exact pattern for uploading a ZIP and then unzipping it in CloudShell.

## Step 4 : Create a Python environment in CloudShell

We will install the Python packages we need for deployment **inside CloudShell**, not on our own computer.

You could potentially do this on your local computer, but you'd need to set up `aws-cli` properly. For a tutorial purposes, I think that is out of the scope, and would just add a layer of complexity. But feel free to try it!

```bash
cd ~/tutorial-work/spam-deploy
python -m venv ~/sm-venv
source ~/sm-venv/bin/activate
pip install --upgrade pip
pip install "sagemaker<3" boto3
python -c "import sagemaker, boto3; print('CloudShell setup OK')"
```

<img src="img/img4_1.png" width="400" height="300">

<img src="img/img4_2.png" width="600" height="300">

#### What this does

- creates a virtual environment called `sm-venv`,
- activates it,
- installs the SageMaker SDK and `boto3`,
- and checks that the installation worked.

CloudShell already includes Python and common development tools, so we only need to install the AWS/Python packages that our tutorial specifically uses. `boto3` is a low-level python SDK for using AWS services inside python scripts.

## Step 5 : Create an S3 bucket

S3 is where we will store files such as:
- the model artifact (`model.tar.gz`)
- the Elastic Beanstalk application ZIP

For this tutorial, you can keep the bucket **private** and make sure it is in the **same Region** as everything else. New S3 buckets block public access by default. This specific bucket and the data inside it is only going to be used by your Sagemaker and EB instance, so no need to make it public.

### Recommended bucket naming rule

To make SageMaker role permissions easier, include the word **`sagemaker`** in the bucket name.

Example:

```text
yourname-sagemaker-spamham-bucket
```
This is kind of important. Because the default IAM role that we create for Sagemaker only allows it to access buckets that have `sagemaker` in their name, you can **[check here](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-roles.html)** for details. Otherwise, you'd need to modify the access for the role/bucket manually.

### In the AWS Console (the main dashboard)

1. Search for **S3**
2. Open the **S3** service
3. Click **Create bucket**
4. Enter a bucket name, (check instructions above, `yourname-sagemaker-spamham-bucket`)
5. Keep the bucket in `ca-central-1`
6. Keep Block all public access turned on
7. Keep all the other settings to default (don't change them)
8. Create the bucket

### After creating the bucket

Open the bucket and create these folders (prefixes):

> - `model-artifacts/`  # will hold the model tarball,
> - `eb-source/`  # will hold the Elastic Beanstalk app ZIP.
> - `data/`  # we are not uploading data here really, but it is a common practice to have a data folder in case you later need to save outputs or something.


<img src="img/img5_1.png" width="400" height="300">

## Step 6 : Create the SageMaker execution role

A **role** is just a permission badge.

The **SageMaker execution role** is the badge SageMaker will use later to:
- read the model artifact from S3
- pull the inference container image
- and write logs/metrics

AWS says SageMaker uses an execution role for operations it performs on your behalf. The simplest starting point is an execution role with the managed policy `AmazonSageMakerFullAccess` attached.

### In the AWS Console

1. Search for **IAM**
2. Open **IAM**
3. In the left sidebar, click **Roles**
4. Click **Create role**
5. For **Trusted entity type**, choose **AWS service**
6. For **Use case**, choose **SageMaker**
7. Choose **SageMaker - Execution**
8. Click **Next**
   - You'll see the `AmazonSageMakerFullAccess` policy attached.
9. Use a simple name like: `sagemaker-inference-role`
10. Create the role!
11. **IMPORTANT**: Save the Role ARN somewhere. You will need it in Session 3.

<img src="img/img6_1.png" width="400" height="300">
<img src="img/img6_2.png" width="400" height="300">

## Step 7 : Create the Elastic Beanstalk service role and EC2 role (computing role)

Elastic Beanstalk needs **two roles**, let's start with the first one.

The **service role** is the permission badge that Elastic Beanstalk itself uses when talking to other AWS services on your behalf. AWS describes it as the role Elastic Beanstalk assumes when it calls services like EC2, Elastic Load Balancing, and Auto Scaling. You can imagine that this is like the permissions you give your building/strata manager to hire cleaners, check alarms, and arrange maintenance.

### In the AWS Console

1. Stay in **IAM**
2. Click **Roles**
3. Click **Create role**
4. Choose **AWS service**
5. Choose **Elastic Beanstalk**
6. Choose the role type / use case for the **Elastic Beanstalk environment**
7. Create the role with the name: `aws-elasticbeanstalk-service-role` (use this unless you have a reason not to!)
8. Make sure these policies are attached
    - `AWSElasticBeanstalkEnhancedHealth`
    - `AWSElasticBeanstalkManagedUpdatesCustomerRolePolicy`
9. Create the role!
10. Save the role name somewhere, as you would need it later.


<img src="img/img7_1.png" width="400" height="300">

<img src="img/img7_2.png" width="400" height="300">

### Now do the exact same thing for the EC2 role

This is the second Elastic Beanstalk role.

AWS says an instance profile is a wrapper around an IAM role that passes role information to an EC2 instance when it starts.

You can think of this as giving the people inside each room of your unit in the appartment permission to use certain supplies and send reports.

In this example usage (for the tutorial), this is the role that will later need permission to call the SageMaker endpoint.

> **STEPS ARE THE SAME AS THE SERVICE ROLE EXCEPT THE FOLLOWING:**
> 
> 6. Choose the role type / use case for the **Elastic Beanstalk – Compute**
> 
> 7. **MAKE SURE YOU NAME THIS ROLE: `aws-elasticbeanstalk-ec2-role`**
>
> 8. Make sure these policies are attached
    - `AWSElasticBeanstalkWebTier`
    - `AWSElasticBeanstalkWorkerTier`
    - `AWSElasticBeanstalkMulticontainerDocker`

### In the AWS Console

1. Search for **VPC**
2. Open the **VPC** service
3. Click **Your VPCs**
4. Look for a VPC with **Default VPC = Yes**

If you already see one, great! You can continue.

If you do **NOT** see a default VPC:
- click **Actions**
- click **Create default VPC**

AWS documents that if you deleted your previous default VPC, you can create a new default VPC, and you can only have one default VPC per Region.

<img src="img/img7_3.png" width="400" height="300">

<img src="img/img7_4.png" width="400" height="300">

## Step 7.5 : Add SageMaker invoke permission to EC2 Role

Because our Flask app will run on the Elastic Beanstalk EC2 instance and call the SageMaker endpoint, this EC2 role needs permission to use the SageMaker action `sagemaker:InvokeEndpoint`. [AWS’s SageMaker authorization reference](https://docs.aws.amazon.com/service-authorization/latest/reference/list_amazonsagemaker.html) lists InvokeEndpoint as an IAM-controlled action and notes that Elastic Beanstalk must have permission to invoke the endpoint.

### How to attach it

After creating the EC2 role:

1. Open the role `aws-elasticbeanstalk-ec2-role` by going to **IAM** -> **Roles** -> searching for the role.
2. Go to the Permissions tab
3. Click **Add permissions**, and choose **Create inline policy**
4. Here, you can either Open the JSON tab, and use the policy in the next cell, **OR**
4. In **Visual**, select `SageMaker`, and set **Action allowed** to `InvokeEndpoint` (by searching and selecting it). Finally, for **Resources** select `All`
5. Name it something clear, for example:
> EBInvokeSageMakerEndpoint

5. Save the policy

<img src="img/img7_5.png" width="400" height="300">

This is the inline policy in JSON format, if you ever wanted to add it in a more painful way! **MAKE SURE YOU MODIFY `YOUR_REGION` AND `YOUR_ACCOUNT_ID` BEFORE USING IT!**
```text
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "AllowInvokeAnyEndpointForTutorial",
      "Effect": "Allow",
      "Action": "sagemaker:InvokeEndpoint",
      "Resource": "arn:aws:sagemaker:YOUR_REGION:YOUR_ACCOUNT_ID:endpoint/*"
    }
  ]
}

## Step 8 : Create a temporary Elastic Beanstalk application and environment

We are going to launch a **sample** Elastic Beanstalk environment now just to prove that the app-hosting side is working.

We are **not** deploying our real spam/ham app yet.

Elastic Beanstalk lets you create an environment using a sample application for the chosen platform. This makes you ready and prepared for **session 3**

### In the AWS Console

1. Search for **Elastic Beanstalk**
2. Open **Elastic Beanstalk**

<img src="img/img8_1.png" width="400" height="300">

3. Click **Create application**
4. Enter:
   - **Application name**: `spam-demo-app`
   - **Environment name**: something like `spam-demo-app-env` (this should appear by default)
   - At the Environment information part, you can make your own domain! Test it out. This is going to be the address typed in the browser to open your app!


<img src="img/img8_2.png" width="400" height="300">

5. Choose:
   - **Web server environment**
   - **Managed platform**
   - **Python**

6. For **application code** choose: **Sample application**

<img src="img/img8_3.png" width="400" height="300">

7. When the wizard asks for roles, choose:
    - **Service role**: your Elastic Beanstalk service role
    - **EC2 instance profile**: your Elastic Beanstalk EC2 role
8. In step 2 of the creation, you don't need to set EC2 keypair, but set it if you want.

<img src="img/img8_4.png" width="400" height="300">

9. In step 3, make sure you click on the appropriate VPC, and the subnet (you can use the public subnet)

<img src="img/img8_5.png" width="400" height="300">

10. **IMPORTANT**: As a free-tier user, you have limitations in types of instances you can select. So make sure in **step 4**, you make the following changes:
    - Root volume: `General Purpose 3(SSD)`, Size `10`GB, IOPS `3000`, Throughput `125`
    - set EC2 security group to default
    - Under Capacity, use the default selections for `Environment type`, `Fleet` and `Architecture`. Remove the `t3.small` from the Instance types, and change `t3.micro` (the first instance type) to `t3a.micro`.

<img src="img/img8_6.png" width="400" height="300">

<img src="img/img8_7.png" width="400" height="300">

11. In **step 5**, set Health monitoring to `Basic`, Disable Managed updates

<img src="img/img8_8.png" width="400" height="300">

12. **IMPORTANT**: in step 5, at the end with setting environment variables, add the following variables, we will need them later:
    - Name: `ENDPOINT_NAME` | Value: `spam-ham-endpoint`
    - Name: `AWS_REGION` | Value: `ca-central-1`

![EB interface](img/img8_9.png)

## Step 9 : Wait for Elastic Beanstalk to turn green

After creation starts, wait until the environment health becomes **OK** or **Green**.

Then open the environment URL (Under `Domain` in EB dashboard) and confirm the sample page loads.

This step is important because it tells you:
- the Elastic Beanstalk roles are correct,
- the Region/networking is okay,
- and the app-hosting side is ready for Session 3.

<img src="img/img9_1.png" width="600" height="300">

## Step 10 : Save the names you will need later

I suggest that you create a notes file in CloudShell (or locally) to keep these:

```bash
cd ~/tutorial-work
cat > session2_notes.txt << 'EOF'
Region:
S3 bucket:
SageMaker role ARN:
EB service role: "aws-elasticbeanstalk-service-role"
EB EC2 instance profile: "aws-elasticbeanstalk-ec2-role"
EB application name:
EB environment name:
EOF
```

This is **NOT** an AWS requirement. It is just a practical way to avoid confusion in Session 3.

## Optional extension : Full deployment walkthrough

In this section, we will go all the way from a saved local model to a working cloud app. To understand and see how things look, I am providing you with the required files. Bear with me and use them as is! I have a surprise for you at the end!

So we will:

1. package the saved model for SageMaker,
2. deploy the model endpoint,
3. zip and package the Elastic Beanstalk app files,
4. deploy the web app,
5. test from both the browser and the terminal.

## Optional Step : Package the model for SageMaker

SageMaker expects the saved model files to be packaged into a `.tar.gz` archive. I have already done this with the spam/ham model, but later when you want to do this with your own model, the archive should contain:

- `model.joblib`
- `metadata.json`

Note: These files need to be at the **root** of the tarball, not inside an extra folder.

You can use the following code to tar your model and its metadata.

In [5]:
from pathlib import Path
import tarfile

model_dir = Path("spam-deploy/model_build/model")
tar_path = Path("spam-deploy/model_build/model.tar.gz")

with tarfile.open(tar_path, "w:gz") as tar:
    for file_path in model_dir.iterdir():
        if file_path.name.startswith("."):
            continue
        tar.add(file_path, arcname=file_path.name)

print(f"Created: {tar_path}")

Created: spam-deploy/model_build/model.tar.gz


In [6]:
with tarfile.open("spam-deploy/model_build/model.tar.gz", "r:gz") as tar:
    print("\n".join(tar.getnames()))

metadata.json
model.joblib


If the output looks like this, you are good:

```text
model.joblib
metadata.json
```
If you see an extra parent folder inside the archive, rebuild it before deploying.

## **IMPORTANT** : Modify your `deploy_sagemaker.py` file

This script is also part of the zip you uploaded to cloudshell at the beginning. But you have to edit it.

To do this, open CloudShell and use `nano` or `vi` to edit the script.

```bash
cd ~/tutorial-work/spam-deploy
source ~/sm-venv/bin/activate
nano deploy_sagemaker.py

At the top of the script, edit:
```
ROLE_ARN

BUCKET
```
If you don't have your IAM Role ARN, you can navigate to `IAM`, `Roles`, and search for `sagemaker`, find the role you created, click on it, and copy the ARN from there.

(If you are using `nano`, use `ctrl`+`x` to save and exit, if you're using `vi`, hit `esc`, then `shift`+`;` (to type `:wq`)
Then run:

Finally, in the CloudShell terminal, run the script!

```bash
python deploy_sagemaker.py


This should take some time to run!

<img src="img/img10_1.png">

## Optional Step : Zip the Flask app (or use my example zipped app)

Elastic Beanstalk’s Python platform supports deploying Python web apps, uses Gunicorn as the default WSGI server, and supports a `Procfile` at the root of the source bundle to define the startup command. A Beanstalk source bundle must be a ZIP file, and it should not include an extra top-level parent folder.

From inside `spam-deploy/eb_app/` from the repo, zip the **contents** of the folder, not the folder itself. This step is different based on what system you use (this happens on your laptop).

So when you open the zip file, you should see files like:
```text
app.py
Procfile
requirements.txt
templates/index.html
```
and **NOT** like this:
```text
eb_app/app.py
eb_app/Procfile
...
```

## Final Step : Deploy the Elastic Beanstalk app

If your account allows Elastic Beanstalk:

1. Recreate or open your Elastic Beanstalk environment
2. Use **Upload and deploy**
3. Upload `eb_app.zip`

## **IMPORTANT** Step 12 : **Terminate** the temporary Elastic Beanstalk environment and your SageMaker Endpoint

We do **NOT** want to leave the sample environment running.

In the Elastic Beanstalk console:

1. Open your environment
2. Click **Actions**
3. Choose **Terminate environment**
4. Confirm

Similarly, we do **NOT** want to leave the SageMaker endpoint running.

In the AWS Console:

1. Search for **Amazon SageMaker AI** and open it
2. In the left panel, click on **Deployments & Interface** -> **Endpoints**
3. Select the endpoint 
4. Click on **Actions** -> **Delete**

AWS says terminating the running environment and endpoints stops you from incurring charges for unused AWS resources, and you can launch a new environment and endpoint again later. This shouldn't take more than a few minutes.

<img src="img/img11_1.png">

## Final checklist for Session 2

Before leaving today, make sure you can answer **yes** to all of these:

- [ ] I launched CloudShell successfully.
- [ ] I uploaded and unzipped my project in CloudShell.
- [ ] I created and tested a Python environment in CloudShell.
- [ ] I created an S3 bucket.
- [ ] I created a SageMaker execution role and saved its ARN.
- [ ] I created an Elastic Beanstalk service role.
- [ ] I created an Elastic Beanstalk EC2 instance profile.
- [ ] I checked that a default VPC exists in my Region.
- [ ] I launched the sample Elastic Beanstalk environment successfully.
- [ ] I used the `deploy_sagemaker.py` script successfully and uploaded my app to EB.
- [ ] I terminated the sample Elastic Beanstalk environment and SageMaker endpoint at the end.

## For next session, it would really help if you set up your EB environment before the tutorial, so we can start with deployment right away!

## Till next session, Adios!